# Lab 4 - Báo cáo mô hình dự đoán giá nhà Boston Housing

Notebook này là báo cáo tổng hợp theo phong cách Lab 2, Lab 3. Notebook chỉ dùng Markdown và hình ảnh sau khi train model, không có code cell.

## 1. Define problem - Định nghĩa bài toán

Bài toán cần giải quyết là dự đoán giá nhà trung vị `MEDV` trong Boston Housing. Đây là bài toán supervised learning dạng regression vì target là giá trị liên tục.

Mục tiêu: cào data, lọc data sạch, chia feature, train model, test model, chọn model tối ưu và dùng model đó trong giao diện Python.

## 2. Hướng giải quyết

Quy trình xử lý gồm: cào raw data từ StatLib, reconstruct data, lọc data sạch, chia feature/target, scale feature, train các model, đánh giá bằng metric, lưu hình ảnh sau train, lưu best model và tạo app dự đoán.

Các model được so sánh: MLP Regression, Linear Regression, Decision Tree Regression, Random Forest Regression.

## 3. Sử dụng những gì

Chương trình sử dụng Python và các thư viện: `pandas`, `numpy`, `scikit-learn`, `tensorflow-cpu`, `matplotlib`, `joblib`, `streamlit`, `pytest`.

Trong đó, `scikit-learn` dùng cho baseline model, scaler và metric; `tensorflow-cpu` dùng cho MLP; `streamlit` dùng cho giao diện Python.

## 4. Data lấy từ đâu và cách lấy data

Nguồn data: Boston Housing raw data từ StatLib/CMU.

`http://lib.stat.cmu.edu/datasets/boston`

Chương trình cào raw text từ URL này và lưu vào `data/raw/boston_raw.txt`. Không dùng `sklearn.datasets.load_boston()` vì API này đã bị loại bỏ.

## 5. Cách lọc data sạch

Dữ liệu raw được lọc theo các bước: bỏ phần mô tả, ghép mỗi mẫu từ 2 dòng raw, chuyển cột sang numeric, xóa duplicate, xóa missing value, kiểm tra schema và kiểm tra target `MEDV` là giá trị dương.

Sau khi lọc, data sạch có 506 dòng và 14 cột.

## 6. Feature và target

Số lượng feature: **13**.

Danh sách feature: `CRIM`, `ZN`, `INDUS`, `CHAS`, `NOX`, `RM`, `AGE`, `DIS`, `RAD`, `TAX`, `PTRATIO`, `B`, `LSTAT`.

Target: `MEDV`, là giá nhà trung vị theo đơn vị nghìn USD.

Biểu đồ skewness giúp nhận biết feature lệch trái hoặc lệch phải để cân nhắc xử lý phân phối trước khi train model.

![Feature skewness left right](../reports/figures/feature_skewness_left_right.png)

Correlation heatmap cho thấy mức tương quan giữa các feature và target `MEDV`, hỗ trợ chọn feature quan trọng và phát hiện feature có tương quan mạnh với nhau.

![Feature correlation heatmap](../reports/figures/feature_correlation_heatmap.png)

## 7. Kết quả so sánh model

Bảng metric trên tập test cho thấy Random Forest Regression đang có RMSE thấp nhất.

**Các bước huấn luyện model:**

1. Lấy dữ liệu sạch, tách `X` gồm 13 feature và `y` là target `MEDV`.
2. Chia dữ liệu thành tập train và tập test với `random_state = 42`.
3. Scale feature bằng `StandardScaler` để các model dùng cùng một đầu vào đã chuẩn hóa.
4. Huấn luyện 4 model trên cùng tập train: MLP Regression, Linear Regression, Decision Tree Regression và Random Forest Regression.
5. Dự đoán trên tập test, tính `MSE`, `RMSE`, `MAE`, `R2`, rồi lưu kết quả vào bảng metric.

**Nhận xét model:** Random Forest Regression có RMSE thấp nhất và R2 cao nhất, nên là model tổng quát tốt nhất trong lần train này. Linear Regression có sai số lớn nhất, cho thấy quan hệ giữa feature và `MEDV` không hoàn toàn tuyến tính.

![Bảng metric so sánh model](../reports/figures/model_metrics_table.png)

## 8. Chọn model tối ưu

Model tối ưu được chọn theo RMSE thấp nhất trên cùng tập test.

**Các bước huấn luyện model:**

1. Sau khi train xong các model, lấy prediction của từng model trên tập test.
2. So sánh RMSE giữa các model bằng bảng metric và biểu đồ cột.
3. Chọn model có RMSE nhỏ nhất làm best model.
4. Lưu best model, scaler và metadata để dùng lại trong app Streamlit.

**Nhận xét model:** Random Forest Regression được chọn vì RMSE = **2.8089**, thấp hơn MLP Regression, Decision Tree Regression và Linear Regression. Điều này phù hợp với dữ liệu tabular nhỏ như Boston Housing, nơi model ensemble dạng cây thường bắt quan hệ phi tuyến tốt.

![So sánh RMSE giữa các model](../reports/figures/model_rmse_comparison.png)

## 9. Hình ảnh sau khi train best model

Random Forest Regression là best model trong lần train này. Hình Actual vs Predicted thể hiện mức độ bám sát giữa giá trị thực và giá trị dự đoán.

**Các bước huấn luyện model:**

1. Khởi tạo `RandomForestRegressor` với `n_estimators = 100` và `random_state = 42`.
2. Fit model trên tập train đã scale.
3. Predict giá nhà trên tập test đã scale.
4. Tính metric và vẽ biểu đồ Actual vs Predicted.
5. Lấy `feature_importances_` để xem feature nào ảnh hưởng nhiều nhất đến dự đoán.

**Nhận xét model:** Các điểm dự đoán của Random Forest nằm khá gần đường lý tưởng, đặc biệt ở vùng giá trung bình. Feature importance cho thấy `RM` và `LSTAT` là hai biến quan trọng nhất, phù hợp với correlation heatmap vì chúng có tương quan mạnh với `MEDV`.

![Actual vs Predicted - Random Forest Regression](../reports/figures/actual_vs_predicted_random_forest_regression.png)

![Feature importance của best model](../reports/figures/best_model_feature_importance.png)

## 10. Hình ảnh sau khi train MLP Regression

MLP Regression được train theo yêu cầu bài toán và được so sánh với các baseline model.

**Các bước huấn luyện model:**

1. Xây dựng mạng neural network gồm input 13 feature, hidden layer 32 neuron, hidden layer 16 neuron và output 1 giá trị dự đoán.
2. Compile model với optimizer `adam`, loss `mean_squared_error` và metric `mae`.
3. Train model trong 100 epoch, batch size 32, dùng tập test làm validation trong lần chạy hiện tại.
4. Predict trên tập test và lưu training curve để xem loss giảm như thế nào.

**Nhận xét model:** MLP học được xu hướng chung của dữ liệu, thể hiện qua RMSE = **3.5520** và R2 = **0.8280**. Tuy nhiên MLP vẫn kém Random Forest, đặc biệt ở một số điểm giá cao model dự đoán thấp hơn thực tế.

![Actual vs Predicted - MLP Regression](../reports/figures/actual_vs_predicted_mlp_regression.png)

![MLP training curve](../reports/figures/mlp_training_curve.png)

## 11. Hình ảnh sau khi train baseline models

Linear Regression và Decision Tree Regression được dùng để làm baseline so sánh với MLP và Random Forest.

**Các bước huấn luyện model:**

1. Train Linear Regression trên tập train đã scale để làm baseline tuyến tính.
2. Train Decision Tree Regression với `random_state = 42` để làm baseline phi tuyến dạng cây đơn.
3. Predict trên cùng tập test cho cả hai baseline.
4. Tính metric và vẽ biểu đồ Actual vs Predicted để so sánh với MLP và Random Forest.

**Nhận xét model:** Linear Regression có RMSE = **4.9286**, cao nhất trong các model, nên chưa mô tả tốt quan hệ phi tuyến của dataset. Decision Tree Regression tốt hơn Linear Regression nhưng vẫn kém Random Forest vì một cây đơn dễ dao động và tổng quát kém hơn ensemble nhiều cây.

![Actual vs Predicted - Linear Regression](../reports/figures/actual_vs_predicted_linear_regression.png)

![Actual vs Predicted - Decision Tree Regression](../reports/figures/actual_vs_predicted_decision_tree_regression.png)

## 12. Kiểm tra lại toàn bộ chương trình

Checklist validation xác nhận chương trình có raw data, clean data, metric, best model, app Python, notebook không có code cell và toàn bộ link hình ảnh đều tồn tại.

![Checklist kiểm tra toàn bộ chương trình](../reports/figures/project_validation_summary.png)

## 13. Kết luận model

Random Forest Regression là model tốt nhất trong lần train hiện tại, với RMSE = **2.8089** và R2 = **0.8924**. MLP Regression đã được train đúng yêu cầu nhưng không phải model tối ưu nhất.

**Các bước huấn luyện model tổng kết:** dữ liệu được làm sạch, chia train/test, scale feature, train nhiều model, dự đoán trên test set, đánh giá bằng MSE/RMSE/MAE/R2, xem biểu đồ dự đoán, xem feature importance, rồi chọn model có RMSE thấp nhất.

**Nhận xét model tổng kết:** Random Forest phù hợp nhất với dataset này vì xử lý tốt quan hệ phi tuyến và cho sai số thấp nhất. MLP vẫn là model hợp lệ nhưng cần tuning thêm nếu muốn cạnh tranh với Random Forest.

Lý do hợp lý là Boston Housing là dataset tabular nhỏ; Random Forest thường hoạt động mạnh trên dữ liệu dạng bảng. Best model đã được lưu tại `models/best_model.joblib` và được dùng trong giao diện Python `app.py`.